# 03. 직접 학습 vs 2단계 학습

용어는 [GLOSSARY.md](../GLOSSARY.md)를 따른다.

NSMC로 **극성 분류**를 먼저 배운 인코더가, 사전학습만 된 인코더보다 KOTE **감정 분류**를 더 잘하는지 비교한다.

| | 직접 학습 | 2단계 학습 |
|---|---|---|
| 인코더 출발점 | 사전학습 KoELECTRA | 01에서 NSMC로 학습한 KoELECTRA의 인코더 |
| 분류 헤드 | 새로 초기화 (8출력) | 새로 초기화 (8출력), **같은 시드면 직접 학습과 같은 초기값** |
| 이후 학습 | KOTE 8범주 다중 라벨, 3 epoch | 동일 |

두 방식의 차이는 **인코더 가중치 하나**뿐이다. 시드 3개(42/43/44)로 돌려 평균 ± 표준편차로 비교한다.

**실행 전에**
- Colab GPU(T4) 런타임에서 실행한다.
- 01 노트북을 실행해 Drive `MyDrive/miniproject/nsmc-polarity`에 NSMC 모델이 있어야 한다.
- 실행마다 결과를 Drive `MyDrive/miniproject/kote-runs/`에 저장한다. 세션이 끊기면 처음부터 다시 실행하면 되고, 이미 끝난 실행은 건너뛴다.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install transformers datasets accelerate scikit-learn

In [ ]:
import os

# SMOKE=1 이면 작은 모델 + 소량 데이터로 코드가 끝까지 도는지만 확인한다.
SMOKE = os.environ.get("SMOKE") == "1"

MODEL_NAME = "monologg/koelectra-small-v3-discriminator" if SMOKE else "monologg/koelectra-base-v3-discriminator"
CATEGORIES = ["기쁨", "슬픔", "분노", "공포", "놀람", "혐오", "지루함", "중립"]
NEUTRAL = CATEGORIES.index("중립")

MAX_LEN = 128       # KOTE 댓글은 64토큰을 넘는 비율이 11%라 NSMC(64)보다 길게 잡는다
EPOCHS = 3
LR = 5e-5
BATCH_SIZE = 64
SEEDS = [42, 43] if SMOKE else [42, 43, 44]
METHODS = ["direct", "two_stage"]
THRESHOLD = 0.5

if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/miniproject"
else:
    DRIVE_DIR = "outputs/smoke" if SMOKE else "outputs"

NSMC_DIR = f"{DRIVE_DIR}/nsmc-polarity"   # 01 노트북이 저장한 극성 분류 모델
RUNS_DIR = f"{DRIVE_DIR}/kote-runs"       # 실행별 지표·확률·모델 (세션이 끊겨도 남도록 Drive)
RESULTS_DIR = "outputs/smoke/results" if SMOKE else "outputs/results"  # git에 커밋할 요약 결과
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1. 데이터

02 노트북이 만든 KOTE 8범주 데이터(`outputs/kote-8cat/`)를 쓴다. 로컬에 없으면(Colab) GitHub에서 읽는다.

In [ ]:
import pandas as pd

KOTE_URL = "https://raw.githubusercontent.com/tlgus2882-ctrl/nlp_miniproject/main/outputs/kote-8cat/{}.csv"

def load_kote(split):
    local = f"outputs/kote-8cat/{split}.csv"
    df = pd.read_csv(local if os.path.exists(local) else KOTE_URL.format(split))
    df["text"] = df["text"].astype(str)
    return df

train_df, val_df, test_df = (load_kote(s) for s in ["train", "val", "test"])

if SMOKE:
    train_df = train_df.sample(200, random_state=42).reset_index(drop=True)
    val_df = val_df.sample(100, random_state=42).reset_index(drop=True)
    test_df = test_df.sample(100, random_state=42).reset_index(drop=True)

print(f"train {len(train_df):,} / val {len(val_df):,} / test {len(test_df):,}")
train_df[CATEGORIES].mean().sort_values(ascending=False).round(3).to_frame("train 비율")

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(df):
    ds = Dataset.from_dict({
        "text": df["text"].tolist(),
        "labels": df[CATEGORIES].values.astype("float32").tolist(),  # BCE 손실은 float 정답을 받는다
    })
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])

train_ds, val_ds, test_ds = (to_dataset(df) for df in [train_df, val_df, test_df])
val_labels = val_df[CATEGORIES].values.astype(int)
test_labels = test_df[CATEGORIES].values.astype(int)

lengths = np.array([len(tokenizer(t)["input_ids"]) for t in train_df["text"]])
print(f"{MAX_LEN}토큰을 넘어 잘리는 train 댓글: {(lengths > MAX_LEN).mean():.1%}")

## 2. 두 방식의 모델 만들기

`AutoModelForSequenceClassification`은 KoELECTRA 체크포인트를 읽어 `ElectraForSequenceClassification`을 만든다.

```
model
├── electra      ← 인코더 (ELECTRA 판별기, Transformer 12층)   ← 2단계 학습은 여기만 NSMC 가중치로 교체
└── classifier   ← 분류 헤드 dense(768→768) → GELU → out_proj(768→8)   ← 두 방식 모두 새로 초기화
```

- `num_labels=8`, `problem_type="multi_label_classification"` → 출력 8개, 손실은 sigmoid + BCE (범주마다 따로 판단)
- 시드를 먼저 고정하고 모델을 만들기 때문에, 같은 시드면 두 방식의 헤드 초기값이 같다
- NSMC 모델의 헤드(긍정/부정 2출력)는 버리고 인코더만 가져온다

In [ ]:
import torch
from transformers import AutoModel, AutoModelForSequenceClassification, set_seed

if not os.path.exists(NSMC_DIR):
    if not SMOKE:
        raise FileNotFoundError(f"{NSMC_DIR} 가 없습니다. 01 노트북을 먼저 실행해 NSMC 모델을 저장하세요.")
    # SMOKE: 01을 돌리지 않아도 저장·불러오기 경로를 확인할 수 있게 작은 극성 모델을 만들어 둔다
    AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).save_pretrained(NSMC_DIR)

def build_model(method, seed):
    set_seed(seed)  # 헤드 초기값을 시드로 고정 → 같은 시드면 두 방식의 헤드가 같다
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(CATEGORIES),
        problem_type="multi_label_classification",
        id2label=dict(enumerate(CATEGORIES)),
        label2id={c: i for i, c in enumerate(CATEGORIES)},
    )
    if method == "two_stage":
        nsmc_encoder = AutoModel.from_pretrained(NSMC_DIR)  # NSMC 모델에서 인코더(electra) 부분만 읽는다
        model.electra.load_state_dict(nsmc_encoder.state_dict())
    return model

In [ ]:
# 확인: 같은 시드에서 두 방식은 헤드가 같고 인코더만 다르다
direct, two_stage = build_model("direct", 42), build_model("two_stage", 42)

def max_diff(a, b):
    return max((pa - pb).abs().max().item() for pa, pb in zip(a.parameters(), b.parameters()))

print("분류 헤드 차이:", max_diff(direct.classifier, two_stage.classifier))
print("인코더 차이   :", max_diff(direct.electra, two_stage.electra))
print(direct.classifier)
del direct, two_stage

## 3. 평가 기준

- 범주마다 확률이 0.5를 넘으면 켠다.
- **배타 규칙**: 중립 외 범주가 하나라도 켜지면 중립을 끈다 (GLOSSARY: 중립은 다른 범주와 함께 붙지 않는다).
- 주 지표는 **macro F1**(8개 범주 F1의 평균)이다. 소수 범주(지루함, 중립)도 똑같은 무게로 반영된다.
- **판정 없음**(아무 범주도 켜지지 않은 댓글) 비율도 함께 본다.
- epoch마다 val macro F1을 재고 가장 좋은 epoch의 모델을 쓴다. test는 고르는 데 쓰지 않는다.

In [ ]:
from sklearn.metrics import f1_score

def apply_exclusive(preds):
    preds = preds.copy()
    other_on = np.delete(preds, NEUTRAL, axis=1).any(axis=1)
    preds[other_on, NEUTRAL] = 0
    return preds

def predict_labels(probs, threshold=THRESHOLD):
    return apply_exclusive((probs > threshold).astype(int))

def score(labels, probs, threshold=THRESHOLD):
    preds = predict_labels(probs, threshold)
    per_cat = f1_score(labels, preds, average=None, zero_division=0)
    return {
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "no_prediction": (preds.sum(axis=1) == 0).mean(),
        **{f"f1_{c}": f for c, f in zip(CATEGORIES, per_cat)},
    }

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return score(labels.astype(int), sigmoid(logits))

## 4. 학습 (2방식 × 시드)

In [ ]:
import gc, json, math, shutil
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

total_steps = math.ceil(len(train_ds) / BATCH_SIZE) * EPOCHS

def run_one(method, seed):
    out_dir = f"{RUNS_DIR}/{method}-seed{seed}"
    if os.path.exists(f"{out_dir}/metrics.json"):
        print(f"[건너뜀] {method} seed {seed}: 이미 결과가 있음")
        return json.load(open(f"{out_dir}/metrics.json"))

    print(f"[학습] {method} seed {seed}")
    model = build_model(method, seed)
    ckpt_dir = f"checkpoints/{method}-seed{seed}"
    args = TrainingArguments(
        output_dir=ckpt_dir,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        warmup_steps=int(total_steps * 0.1),
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,   # val macro F1이 가장 좋은 epoch으로 되돌린다
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        report_to="none",
        seed=seed,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )
    trainer.train()

    evals = [h for h in trainer.state.log_history if "eval_macro_f1" in h]
    best_epoch = max(evals, key=lambda h: h["eval_macro_f1"])["epoch"]

    val_probs = sigmoid(trainer.predict(val_ds).predictions)
    test_probs = sigmoid(trainer.predict(test_ds).predictions)
    metrics = {
        "method": method, "seed": seed, "best_epoch": best_epoch,
        **{f"val_{k}": float(v) for k, v in score(val_labels, val_probs).items()},
        **{f"test_{k}": float(v) for k, v in score(test_labels, test_probs).items()},
    }

    os.makedirs(out_dir, exist_ok=True)
    np.save(f"{out_dir}/val_probs.npy", val_probs)    # 04에서 threshold 튜닝에 쓴다
    np.save(f"{out_dir}/test_probs.npy", test_probs)
    if method == "two_stage":                        # 최종 모델 후보만 저장 (04에서 시드 선택)
        trainer.save_model(f"{out_dir}/model")
        tokenizer.save_pretrained(f"{out_dir}/model")
    json.dump(metrics, open(f"{out_dir}/metrics.json", "w"), ensure_ascii=False, indent=1)  # 마지막에 써야 '끝난 실행'으로 인정된다

    shutil.rmtree(ckpt_dir, ignore_errors=True)
    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"  best epoch {best_epoch:g} | val macro F1 {metrics['val_macro_f1']:.4f} | test macro F1 {metrics['test_macro_f1']:.4f}")
    return metrics

In [ ]:
rows = [run_one(method, seed) for method in METHODS for seed in SEEDS]

runs = pd.DataFrame(rows)
runs.to_csv(f"{RESULTS_DIR}/runs.csv", index=False)
runs.to_csv(f"{RUNS_DIR}/runs.csv", index=False)  # Colab 로컬 파일은 세션과 함께 사라지므로 Drive에도
print(f"저장: {RESULTS_DIR}/runs.csv (Colab이면 Drive {RUNS_DIR}/runs.csv 를 내려받아 저장소 outputs/results/ 에 커밋)")
runs[["method", "seed", "best_epoch", "val_macro_f1", "test_macro_f1", "test_micro_f1", "test_no_prediction"]].round(4)

## 5. 결과 비교

In [ ]:
METHOD_NAMES = {"direct": "직접 학습", "two_stage": "2단계 학습"}

def mean_std(s):
    return f"{s.mean():.4f} ± {s.std(ddof=1):.4f}" if len(s) > 1 else f"{s.mean():.4f}"

cols = ["val_macro_f1", "test_macro_f1", "test_micro_f1", "test_no_prediction"]
summary = runs.groupby("method")[cols].agg(mean_std).rename(index=METHOD_NAMES)
summary

In [ ]:
# 범주별 test F1 (시드 평균): 2단계 학습이 어느 범주에서 이기고 지는지
per_cat = runs.groupby("method")[[f"test_f1_{c}" for c in CATEGORIES]].mean().T
per_cat.index = CATEGORIES
per_cat = per_cat.rename(columns=METHOD_NAMES)
per_cat["차이 (2단계 − 직접)"] = per_cat["2단계 학습"] - per_cat["직접 학습"]
per_cat.round(4)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False
if IN_COLAB:
    !apt-get -qq install fonts-nanum > /dev/null
    import matplotlib.font_manager as fm
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rcParams["font.family"] = "NanumGothic"

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(CATEGORIES))
for i, (name, color) in enumerate([("직접 학습", "#8a94a6"), ("2단계 학습", "#d9622b")]):
    ax.bar(x + (i - 0.5) * 0.38, per_cat[name], width=0.38, label=name, color=color)
ax.set_xticks(x, CATEGORIES)
ax.set_ylabel("test F1 (시드 평균)")
ax.set_ylim(0, 1)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 6. 해석 기준

- 두 방식의 test macro F1 평균 차이가 **표준편차보다 작으면** "차이가 있다고 보기 어렵다"고 쓴다. 시드 3개는 통계 검정을 하기엔 적다.
- 2단계 학습이 지더라도 최종 모델은 2단계 학습이다 (GLOSSARY). 그때는 범주별 표에서 어느 범주가 손해를 봤는지를 결론으로 쓴다.
  - 예: 극성과 직접 이어지는 범주(기쁨·분노)는 이득, 극성으로 구분되지 않는 범주(놀람·공포·중립)는 손해
- 한계: 2단계 학습 3시드는 모두 같은 NSMC 모델(시드 42)에서 출발하므로, NSMC 학습 자체의 변동은 표준편차에 들어 있지 않다.

다음: **04 노트북**에서 2단계 학습 3시드 중 val macro F1이 가장 높은 모델을 최종 모델로 고르고, 범주별 threshold 튜닝과 데모를 한다.